# Recursive guarded world model

**Research question:** can a machine learn a reusable operation, change its conditions of applicability, and reinterpret older experience in the resulting vocabulary—faster than WARC learns through weight updates alone?

Read [the MechEval paper](https://arxiv.org/abs/2611.12324) and [WARC source](https://github.com/third-intelligence/WARC/tree/f0081ef00e867f63a08b1e4e7350d2baa0440267). WARC is a 35M pretrained action-conditioned Transformer, 3 frames, 10 blocks, 512 width, 2x2 patches. Its released online recipe uses 20 updates after each valid transition, 10 recent + 20 old transitions, rotations, Muon/AdamW.

**Prediction before running:** the scratch learner may lose initially. The frozen-WARC + growing rule learner is our highest-upside practical bet: start with pretrained knowledge, learn a cheap transferable correction, then compare with full WARC TTT and the same pretrained prior with a direct prediction head. If transfer-direct wins, the recursive mechanism has not earned its complexity. If stale and small win, reject retrospective revision or growth respectively.

**Scope:** this is a hypothesis-testing pilot, not a SOTA result. Official MechEval mechanic annotations/scorer are unreleased in the pinned upstream README. The live benchmark uses real human ARC transitions, changed-cell accuracy, exact frames and false edits; those are NOT the paper's per-mechanic metric.

## 1 · Architecture (preserve this contract)

WARC changes its weights in response to new evidence. Our intended architecture changes the **language of computation** in addition to weights.

At time t, observations and actions produce a state representation. A guard chooses a label conditioned on this representation and prior label field. A separate dictionary maps that label to an executable effect. Execution transforms the canvas, whose result and label field become inputs to the next pass. On observation of the true outcome, the model assigns retrospective responsibilities to old transitions and splits a high-error label when appropriate. Replaying old observations with the revised dictionary can change what they mean.

The current implementation is deliberately bounded: 3 passes, 4→12 active slots, a supplied local copy/write basis, learned guard and effect mixtures, approximate posterior relabeling. The full hypothesis additionally requires **learned operator composition as first-class reusable primitives**, learned metarules controlling other rules, and persistent reconstruction of earlier explanations. These are research milestones, not features implemented or proved here. No architectural change should quietly remove guard/effect separation, re-interpretation or executable composition. Transformer encoders and transfer learning are compatible with the hypothesis.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(14, 8), facecolor='white')
def node(ax, xy, label, color='#eaf1f7', w=.76, h=.085):
    x,y=xy
    ax.add_patch(FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=.012',
                 facecolor=color,edgecolor='#40576a',linewidth=1.15))
    ax.text(x,y,label,ha='center',va='center',fontsize=10,color='#182b39')
def arrow(ax,a,b,color='#40576a',rad=0,ls='-'):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=13,
                color=color,connectionstyle=f'arc3,rad={rad}',linestyle=ls,linewidth=1.4))
for ax in axes:
    ax.set(xlim=(0,1),ylim=(0,1)); ax.axis('off')
a,b=axes
a.set_title('WARC — original',fontsize=15,weight='bold')
for y,t in [( .90,'3 frames + 3 actions'),(.75,'2×2 patch tokens + click marker'),
            (.60,'10 action-conditioned Transformer blocks'),(.45,'16-color next-frame head'),
            (.30,'Prediction → observe actual next frame'),(.12,'Replay → 20 gradient updates')]:
    node(a,(.47,y),t)
for y,z in zip([.90,.75,.60,.45,.30],[.75,.60,.45,.30,.12]):
    arrow(a,(.47,y-.045),(.47,z+.045))
arrow(a,(.87,.12),(.87,.60),'#a56120',-.15)
a.text(.93,.36,'Update weights',rotation=90,ha='center',va='center',fontsize=9,color='#a56120')
b.set_title('Ours — implemented pilot + transfer',fontsize=15,weight='bold')
node(b,(.46,.90),'3 frames + actions + optional frozen WARC prior')
node(b,(.46,.75),'Encoder + hidden state + previous label field')
node(b,(.26,.60),'Guard: where / when','#def3ee',w=.37)
node(b,(.69,.60),'Dictionary: effect','#def3ee',w=.37)
node(b,(.46,.45),'Execute on intermediate canvas','#def3ee')
node(b,(.46,.30),'Final prediction (optional learned prior mixture)')
node(b,(.46,.12),'Observe → replay → relabel / split 4→12','#f9ecd8')
arrow(b,(.46,.855),(.46,.795)); arrow(b,(.46,.705),(.26,.645))
arrow(b,(.26,.555),(.40,.495)); arrow(b,(.69,.555),(.54,.495))
arrow(b,(.46,.405),(.46,.345)); arrow(b,(.46,.255),(.46,.165))
arrow(b,(.86,.45),(.86,.75),'#147e78',-.19)
b.text(.93,.60,'3 execution passes',rotation=90,ha='center',fontsize=9,color='#147e78')
arrow(b,(.86,.12),(.85,.60),'#a56120',-.19,'--')
b.text(.92,.31,'Revise dictionary',rotation=90,ha='center',fontsize=9,color='#a56120')
fig.tight_layout(); plt.show()

## 2 · Set up (Colab or Kaggle)

Select GPU; enable Internet on Kaggle. L4/A100 preferred. T4 uses a logged FP32 fallback for upstream WARC (including Muon). The notebook uses one GPU. Run cells in order; failed commands raise immediately. Source is pinned to the audited commit, not a moving branch.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator and reconnect.'
print(torch.cuda.get_device_name(0), torch.__version__, 'BF16:',torch.cuda.is_bf16_supported())
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content')/'recursive_research'
WORK.mkdir(parents=True,exist_ok=True)
def run(*args):
    subprocess.run([str(x) for x in args],check=True)
REPO=WORK/'fastai'
if not (REPO/'.git').exists():
    run('git','clone','--depth','1','--branch','research/recursive-world-model',
        'https://github.com/shekhawatxyz/fastai-2022',REPO)
run('git','-C',REPO,'fetch','--depth','1','origin','2529f513d3eb68b26295d9ddf69d4346b2a0e4c4')
run('git','-C',REPO,'checkout','--detach','2529f513d3eb68b26295d9ddf69d4346b2a0e4c4')
SCRIPT=REPO/'experiments/recursive_world/recursive_world.py'
run(sys.executable,'-m','py_compile',SCRIPT)

## 3 · Prove the plumbing first

Predict: normalized probabilities, causal inference, hard execution, nonzero weight updates, label birth, and transfer smoke tests must pass. Synthetic color transfer is a sanity test, not evidence of real-world superiority.

In [ ]:
run(sys.executable,SCRIPT,'--suite','smoke','--epochs','4','--seeds','1',
    '--arms','direct,copy,small,stale,grow','--profile','--out',WORK/'smoke')

## 4 · Real WARC comparison

The public dataset mirror is a convenient default; official recording byte identity is unverified. For definitive public-data experiments, set DATA to the extracted [ARC Prize Public Demo dataset](https://arcprize.org/blog/arc-agi-3-human-dataset) folder and check WARC's data SHA256SUMS. Both models get exactly the same causal stream. No model sees the target before prediction. Fold checkpoints exclude the tested game. Each recording starts with fresh weights; this is a replay proxy, not the official MechEval protocol.

Predict: the transfer model should initially track the frozen checkpoint. If gains come only from transfer rather than rule induction, the identical direct-transfer control will expose this. Evaluate quality and speed, including the separate teacher-cache cost.

In [ ]:
WARC=WORK/'WARC'
if not (WARC/'.git').exists():
    run('git','clone','https://github.com/third-intelligence/WARC',WARC)
run('git','-C',WARC,'checkout','f0081ef00e867f63a08b1e4e7350d2baa0440267')
run(sys.executable,'-m','pip','install','-q','datasets','einops','timm','huggingface_hub',
    'arc-agi==0.9.9','arcengine==0.9.3')
DATA=None # Set to official extracted directory if available.
base=[sys.executable,SCRIPT,'--suite','arc','--warc-root',WARC]
if DATA: base += ['--data',DATA]
# 24 valid transitions, one recording: integration pilot, NOT a scientific result.
run(*base,'--games','ls20','--sessions','1','--limit','24','--seeds','1',
    '--arms','direct,copy,grow','--transfer','--transfer-arms','direct,grow',
    '--warc-arms','frozen,ttt,scratch','--profile','--out',WORK/'pilot')

## 5 · Read the results before increasing compute

Primary metric: changed-cell accuracy averaged per recording, then game. Exact-frame and false-edit rates prevent a model from winning by painting everything. Time and number of examples/updates are separate, not silently equated. A fixed 20-step budget matches the published WARC update count, not FLOPs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
def report(folder):
    p=WORK/folder
    scores=pd.read_csv(p/'scores.csv')
    cols=['changed_acc','exact','false_edit','change_f1']
    paired=scores.groupby(['game','session','arm','seed'])[cols].mean().groupby(['game','session','arm']).mean()
    display(paired.groupby(['game','arm']).mean().round(4))
    display(scores.groupby('arm')[['predict_s','update_s','teacher_cache_s']].mean().round(4))
    scores.groupby(['step','arm']).changed_acc.mean().unstack().rolling(4,min_periods=1).mean().plot(
        figsize=(11,4),ylim=(0,1),title='Causal replay: changed-cell accuracy')
    plt.show()
    drift=pd.read_csv(p/'weight_drift.csv')
    drift[drift.arm=='transfer_grow'].pivot_table(index='epoch',columns='parameter',values='relative_init').plot(
        legend=False,title='Transfer-grow: per-tensor drift from initialization',figsize=(11,4))
    plt.show()
    display(pd.read_csv(p/'vocabulary.csv').query('arm == "transfer_grow" and active == True').tail(24))
    if (p/'profile.txt').exists(): print((p/'profile.txt').read_text())
    print('Artifacts:',p)
    return paired
pilot=report('pilot')

## 6 · Scale only after pilot passes

Frozen configuration: 128 causal exposures per recording (or fewer if exhausted), 2 recordings per game, 3 seeds. Run the 25 public games by fold. Do not tune on these results and then call them untouched; reserve independent games/recordings for confirmation. The run is expensive; start one fold/game at a time if GPU quota is limited. More exposures can be tested after checking replay memory. This is a fixed-budget public replay evaluation, not the private ARC competition or the unavailable annotated MechEval scorer.

**Acceptance:** transfer-grow must beat WARC TTT on the same transitions, not only the frozen model; improve exact accuracy without materially increasing false edits; compare with transfer-direct, transfer-stale, transfer-small, transfer-fixed. Report paired game-level intervals, not independent-cell p-values. If only the ensemble wins, attribute the extra compute explicitly. Don't change five things at once: first test this, then try pretraining our own encoder, better operator primitives, or learned metarules.

In [ ]:
# Uncomment only when the pilot completes. Set separate output directories per fold.
# run(*base,'--games','all','--sessions','2','--limit','128','--seeds','3',
#     '--arms','direct,copy,grow','--transfer',
#     '--transfer-arms','direct,fixed,small,stale,grow',
#     '--warc-arms','frozen,ttt,scratch','--profile','--out',WORK/'full')
# full=report('full')

In [ ]:
# Paired game-cluster bootstrap on the complete sweep; no significance claim on a one-game pilot.
import numpy as np
def paired_delta(paired, candidate='transfer_grow', baseline='WARC ttt'):
    game=paired['changed_acc'].unstack('arm').groupby('game').mean().dropna(subset=[candidate,baseline])
    delta=(game[candidate]-game[baseline]).to_numpy()
    assert len(delta)>=10, 'Too few independent games for a useful interval.'
    rng=np.random.default_rng(42)
    boot=rng.choice(delta,(10000,len(delta)),replace=True).mean(1)
    return dict(games=len(delta),difference_pp=100*delta.mean(),ci95_pp=(100*np.quantile(boot,[.025,.975])).tolist())
# print(paired_delta(full))

## Published reference only — MechEval

| Method | All mechanic accuracy | Exposures to criterion | Not reached |
|---|---:|---:|---:|
| WARC frozen | 18.4% | 114 | 85.5% |
| WARC GRU | 24.0% | 87 | 84.1% |
| DreamerV3 | 25.9% | 114 | 90.8% |
| Episodic retrieval | 32.9% | 109 | 76.8% |
| WARC scratch | 35.0% | 105 | 51.2% |
| WARC TTT | 69.1% | 74 | 18.8% |

Source: [upstream evaluation table](https://github.com/third-intelligence/WARC#evaluation). Never compare a live replay score directly with these numbers. The actual baseline runs in section 4.

**Development status:** source reviewed against upstream; GPU integration not executed in the authoring environment. This is an executable research candidate with built-in tests, not a verified result.